# A/B-тест новой формы оплаты ЖКХ

Цель — сравнить конверсию в успешную оплату между группами A и B. Анализ ведётся **на уровне платежей**; если один пользователь совершал несколько платежей, независимость наблюдений необходимо проверить отдельно.

## 1. Загрузка данных

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Файл Data.xlsx должен лежать рядом с этим ноутбуком.
DATA_FILE = Path("Data.xlsx")
if not DATA_FILE.exists():
    raise FileNotFoundError("Поместите Data.xlsx в ту же папку, что и Work.ipynb")

users = pd.read_excel(DATA_FILE, sheet_name="Пользователи")
payments = pd.read_excel(DATA_FILE, sheet_name="Платежи")
print(f"Пользователи: {len(users)}, платежи: {len(payments)}")

In [ ]:
display(users.head())
display(payments.head())

## 2. Проверка и очистка данных

In [ ]:
print("Пользователи:")
users.info()
print("Пропуски:\n", users.isnull().sum())
print("Повторные user_id:", users.duplicated(subset="user_id").sum())
print("Полные дубликаты:", users.duplicated().sum())

In [ ]:
users_clean = users.copy()
min_age, max_age = 20, 100
users_clean = users_clean.query("age >= @min_age and age <= @max_age")
users_clean = users_clean.dropna(subset=["city", "group"])
users_clean = users_clean.drop_duplicates(subset="user_id")
print("Пользователей после очистки:", len(users_clean))

In [ ]:
print("Платежи:")
payments.info()
print("Пропуски:\n", payments.isnull().sum())
print("Повторные payment_id:", payments.duplicated(subset="payment_id").sum())
print("Полные дубликаты:", payments.duplicated().sum())

In [ ]:
payments_clean = payments.copy()
payments_clean = payments_clean.dropna(subset=["user_id"])
user_group_map = users_clean.set_index("user_id")["group"].to_dict()
payments_clean["group"] = payments_clean["user_id"].map(user_group_map)

users_clean = users_clean.sort_values("user_id").reset_index(drop=True)
payments_clean = payments_clean.sort_values("user_id").reset_index(drop=True)
print("Платежей после первичной очистки:", len(payments_clean))

## 3. Объединение таблиц

In [ ]:
merged = pd.merge(users_clean, payments_clean, on="user_id", how="inner")
merged["success"] = np.where(merged["step4_success"].notna(), "+", "-")
merged = merged.drop(columns="group_y").rename(columns={"group_x": "group"})
merged = merged[merged["group"].isin(["A", "B"])].copy()
print("Платежей для анализа:", len(merged))
display(merged.head())

In [ ]:
# Необязательно: сохранить результат очистки в отдельный Excel-файл.
with pd.ExcelWriter("cleanData.xlsx") as writer:
    users_clean.to_excel(writer, sheet_name="Пользователи", index=False)
    payments_clean.to_excel(writer, sheet_name="Платежи", index=False)
    merged.to_excel(writer, sheet_name="Объединенная", index=False)

## 4. Конверсия и статистическая проверка

Проверяем двустороннюю гипотезу о равенстве конверсий групп. Для Z-теста используется объединённая доля успехов; для доверительного интервала разности — отдельные оценки дисперсий групп (интервал Вальда).

In [ ]:
n_A = len(merged[merged["group"] == "A"])
n_B = len(merged[merged["group"] == "B"])
success_A = (merged.loc[merged["group"] == "A", "success"] == "+").sum()
success_B = (merged.loc[merged["group"] == "B", "success"] == "+").sum()

p_A = success_A / n_A
p_B = success_B / n_B
p = (success_A + success_B) / (n_A + n_B)
se_test = np.sqrt(p * (1 - p) * (1 / n_A + 1 / n_B))
z_stat = (p_B - p_A) / se_test
p_value = 2 * stats.norm.sf(abs(z_stat))

se_ci = np.sqrt(p_A * (1 - p_A) / n_A + p_B * (1 - p_B) / n_B)
lower = (p_B - p_A) - 1.96 * se_ci
upper = (p_B - p_A) + 1.96 * se_ci

print(f"Группа A: {n_A} платежей, успешно: {success_A} ({p_A:.1%})")
print(f"Группа B: {n_B} платежей, успешно: {success_B} ({p_B:.1%})")
print(f"Разница: {(p_B - p_A) * 100:.1f} п.п.")
print(f"Z-статистика: {z_stat:.3f}")
print(f"P-value: {p_value:.3g}")
print(f"95% ДИ разности: [{lower * 100:.1f}; {upper * 100:.1f}] п.п.")
print(f"Статистически значимо при α=0.05: {'ДА' if p_value < 0.05 else 'НЕТ'}")

## 5. Визуализация результатов

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["A (контроль)", "B (тест)"], [p_A * 100, p_B * 100])
ax.set_ylabel("Конверсия, %")
ax.set_title("Конверсия в успешную оплату")
ax.set_ylim(0, max(p_A, p_B) * 115)
for bar, value in zip(bars, [p_A, p_B]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{value:.1%}", ha="center")
plt.tight_layout()
plt.show()

## 6. Вывод

Сравнение показывает разницу в конверсии между группами. Перед рекомендацией о запуске на всех пользователей следует проверить, что распределение в A/B-группы было корректным, а повторные платежи одного пользователя не нарушают предпосылку независимости наблюдений.